# RAG Pipeline Walkthrough

This notebook demonstrates the complete Retrieval-Augmented Generation (RAG) pipeline step by step, using the "Attention Is All You Need" paper (the original Transformer paper) as the example document.

**Goal**: Understand each component of a RAG system by running it piece by piece:
1. Load and extract text from a PDF
2. Chunk the text with sentence overlap
3. Embed chunks using BAAI/bge-small-en-v1.5
4. Build a FAISS vector index
5. Retrieve relevant chunks for a query
6. Generate an answer using flan-t5-base

This mirrors what the Streamlit app (`app.py`) does internally, but exposed for learning and debugging.

In [ ]:
# Setup: Add project root to path and import utilities
import sys
import os
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(''), '..'))
sys.path.insert(0, PROJECT_ROOT)

from rag_utils import (
    extract_text_from_pdf,
    chunk_text,
    build_faiss_index,
    embed_query,
    load_embedding_model,
    process_pdf,
)
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import faiss
import numpy as np

PDF_PATH = os.path.join(PROJECT_ROOT, "pdf_for_upload_test.pdf")
print(f"PDF path: {PDF_PATH}")
print(f"Exists: {os.path.exists(PDF_PATH)}")

: 

## 1. Load and Extract Text from PDF

We use PyPDF2 to extract text page by page, preserving page numbers for citations.

In [ ]:
pages_text = extract_text_from_pdf(PDF_PATH)
print(f"Number of pages: {len(pages_text)}")

# Show first page text sample
page_num, text = pages_text[0]
print(f"\nPage {page_num} (first 500 chars):")
print(text[:500])

## 2. Chunk the Text

We split the text into overlapping chunks (by sentence) to ensure context isn't cut mid-thought. Each chunk tracks its starting page number for citation.

In [ ]:
CHUNK_SIZE = 500
OVERLAP_SENTENCES = 2

chunks, chunk_pages = chunk_text(pages_text, chunk_size=CHUNK_SIZE, overlap_sentences=OVERLAP_SENTENCES)

print(f"Number of chunks: {len(chunks)}")
print(f"Chunk size target: {CHUNK_SIZE} chars")
print(f"Overlap sentences: {OVERLAP_SENTENCES}")

# Show a few example chunks
for i in [0, 1, 10, 50]:
    if i < len(chunks):
        print(f"\n--- Chunk {i} (page {chunk_pages[i]}) ---")
        print(f"Length: {len(chunks[i])} chars")
        print(chunks[i][:300] + "...")

## 3. Embed the Chunks with BAAI/bge-small-en-v1.5

BGE models benefit from a query instruction prefix ("Represent this sentence for searching relevant passages: ") for retrieval tasks. We apply this **only to queries**, not to document chunks.

We also use `normalize_embeddings=True` so that FAISS inner product equals cosine similarity.

In [ ]:
embed_model = load_embedding_model()
print(f"Model loaded: {embed_model}")

index, embeddings = build_faiss_index(chunks, embed_model)

print(f"\nEmbedding shape: {embeddings.shape}")
print(f"Embedding dimension: {embeddings.shape[1]}")
print(f"FAISS index type: {type(index).__name__}")
print(f"Index total vectors: {index.ntotal}")

# Verify normalization (should be ~1.0 for normalized vectors)
norms = np.linalg.norm(embeddings, axis=1)
print(f"\nEmbedding norms - min: {norms.min():.4f}, max: {norms.max():.4f}, mean: {norms.mean():.4f}")

## 4. FAISS Index is Built

The index is already built in the previous step (`build_faiss_index` returns the index). It uses `IndexFlatIP` (inner product) on normalized vectors, which is equivalent to cosine similarity search.

## 5. Run an Example Query End-to-End

Now we'll embed a query (with the BGE instruction prefix), search the index, and show the retrieved chunks with similarity scores and page numbers.

In [ ]:
QUESTION = "How does multi-head attention work?"
TOP_K = 3

print(f"Query: {QUESTION}")

# Embed query with BGE instruction prefix
q_emb = embed_query(embed_model, QUESTION)
print(f"Query embedding shape: {q_emb.shape}")

# Search
D, I = index.search(q_emb, k=TOP_K)

print(f"\nTop {TOP_K} results:")
for rank, (idx, score) in enumerate(zip(I[0], D[0])):
    if idx == -1:
        continue
    snippet = chunks[idx][:250]
    print(f"\n  Rank {rank+1}: page {chunk_pages[idx]}, similarity: {score:.4f}")
    print(f"  {snippet}...")

## 6. Generate an Answer with flan-t5-base

We take the retrieved chunks as context, format a prompt, and generate an answer using the flan-t5-base model.

In [ ]:
# Load flan-t5-base
print("Loading flan-t5-base...")
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
model.eval()
print("Model loaded.")

# Build context from retrieved chunks
context = "\n".join(chunks[i] for i in I[0])
prompt = f"Question: {QUESTION}\nContext: {context}\nAnswer:"

print(f"\nPrompt length: {len(prompt)} chars")
print(f"Prompt preview:\n{prompt[:500]}...")

# Generate answer
inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
outputs = model.generate(
    **inputs,
    max_length=200,
    num_beams=4,
    repetition_penalty=1.3,
    no_repeat_ngram_size=3,
    early_stopping=True,
)
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"\n{'='*60}")
print(f"GENERATED ANSWER:")
print(f"{'='*60}")
print(answer)

## 7. Reflection

### What worked well:
- **BGE embeddings** with the query instruction prefix produce strong semantic matches — the top results for "multi-head attention" are precisely the sections explaining that mechanism (pages 4-5).
- **Sentence-overlap chunking** preserves context across boundaries, so key explanations aren't split awkwardly.
- **FAISS IndexFlatIP** on normalized vectors gives exact cosine similarity — fast enough for this scale (128 chunks) and no approximation error.
- **flan-t5-base** generates coherent, grounded answers when given relevant context.

### The relevance threshold (0.3 in the app):
- This is a cosine-similarity floor. If the best retrieved chunk scores below 0.3, the app refuses to answer rather than hallucinate.
- In this run, the best score was ~0.82 — well above threshold. For out-of-domain questions, scores typically fall below 0.3.
- The threshold is a **safety guard**, not a quality optimizer. Setting it too high rejects valid answers; too low invites hallucination.

### Limitations observed:
1. **Chunk size trade-off**: Smaller chunks (300) give more granular retrieval but more fragments; larger chunks (800) capture more context but dilute focus. The evaluation script (`scripts/evaluate_chunking.py`) compares this systematically.
2. **No reranking**: We retrieve top-k by embedding similarity only. A cross-encoder reranker would improve precision but adds latency.
3. **Single-vector per chunk**: Long chunks may contain multiple topics; late-interaction models (ColBERT) or multi-vector approaches could help.
4. **flan-t5-base is small** (250M params). Larger models (flan-t5-large, Llama, etc.) generate better answers but need more RAM/VRAM.
5. **No citation enforcement in generation**: The model may not explicitly cite sources. The app shows sources separately.
6. **PDF extraction quality**: Scanned/image-only PDFs fail. PyPDF2 struggles with complex layouts.

### Next steps for production:
- Add embedding/FAISS caching (pickle) to avoid re-processing on restart
- Implement hybrid search (BM25 + dense) for keyword-heavy queries
- Add a cross-encoder reranker for higher precision
- Support more document types (DOCX, HTML, plain text)
- Add streaming token output for better UX